# 08 - Window Functions & Ranking on NYC Taxi Data

Extends the groupby/join pattern from notebook 07 with **window functions** (ROW_NUMBER, RANK, LAG) to analyse revenue trends by hour and zone.

## 1. Initialize Spark Session

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .master("local[*]") \
    .appName('window_functions') \
    .getOrCreate()

## 2. Load and Register Parquet Data

In [ ]:
df_green = spark.read.parquet('data/pq/green/*/*')
df_green.registerTempTable('green')

df_yellow = spark.read.parquet('data/pq/yellow/*/*')
df_yellow.registerTempTable('yellow')

## 3. Compute Revenue by Hour and Zone using SQL

In [ ]:
df_green_revenue = spark.sql("""
SELECT 
    date_trunc('hour', lpep_pickup_datetime) AS hour, 
    PULocationID AS zone,
    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    green
WHERE
    lpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

In [ ]:
df_yellow_revenue = spark.sql("""
SELECT 
    date_trunc('hour', tpep_pickup_datetime) AS hour, 
    PULocationID AS zone,
    SUM(total_amount) AS amount,
    COUNT(1) AS number_records
FROM
    yellow
WHERE
    tpep_pickup_datetime >= '2020-01-01 00:00:00'
GROUP BY
    1, 2
""")

## 4. Write Revenue Reports to Parquet

In [ ]:
df_green_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/green', mode='overwrite')

df_yellow_revenue \
    .repartition(20) \
    .write.parquet('data/report/revenue/yellow', mode='overwrite')

## 5. Rename Columns for Join Clarity

In [ ]:
df_green_revenue = spark.read.parquet('data/report/revenue/green')
df_yellow_revenue = spark.read.parquet('data/report/revenue/yellow')

df_green_revenue_tmp = df_green_revenue \
    .withColumnRenamed('amount', 'green_amount') \
    .withColumnRenamed('number_records', 'green_number_records')

df_yellow_revenue_tmp = df_yellow_revenue \
    .withColumnRenamed('amount', 'yellow_amount') \
    .withColumnRenamed('number_records', 'yellow_number_records')

## 6. Outer Join Green and Yellow Revenue

In [ ]:
df_join = df_green_revenue_tmp.join(df_yellow_revenue_tmp, on=['hour', 'zone'], how='outer')

df_join.write.parquet('data/report/revenue/total', mode='overwrite')

## 7. Window Functions — Rank Zones by Revenue per Hour

Use `Window` partitioned by `hour` to rank zones and compute running totals and lag comparisons.

In [ ]:
df_join = spark.read.parquet('data/report/revenue/total')

# Window partitioned by hour, ordered by combined revenue descending
window_hour = Window.partitionBy('hour').orderBy(
    F.desc(F.coalesce(F.col('green_amount'), F.lit(0)) + F.coalesce(F.col('yellow_amount'), F.lit(0)))
)

df_ranked = df_join \
    .withColumn('total_amount', 
                F.coalesce(F.col('green_amount'), F.lit(0)) + F.coalesce(F.col('yellow_amount'), F.lit(0))) \
    .withColumn('rank',        F.rank().over(window_hour)) \
    .withColumn('row_number',  F.row_number().over(window_hour)) \
    .withColumn('dense_rank',  F.dense_rank().over(window_hour))

df_ranked.select('hour', 'zone', 'total_amount', 'rank', 'row_number', 'dense_rank') \
    .orderBy('hour', 'rank') \
    .show(20, truncate=False)

In [ ]:
# LAG: compare each zone's revenue to the previous hour's revenue (ordered by hour per zone)
window_zone = Window.partitionBy('zone').orderBy('hour')

df_lagged = df_join \
    .withColumn('total_amount',
                F.coalesce(F.col('green_amount'), F.lit(0)) + F.coalesce(F.col('yellow_amount'), F.lit(0))) \
    .withColumn('prev_hour_amount', F.lag('total_amount', 1).over(window_zone)) \
    .withColumn('revenue_change',   F.col('total_amount') - F.col('prev_hour_amount'))

df_lagged.select('zone', 'hour', 'total_amount', 'prev_hour_amount', 'revenue_change') \
    .orderBy('zone', 'hour') \
    .show(20, truncate=False)

## 8. Enrich with Zone Lookup Data

In [ ]:
df_zones = spark.read.option("header", "true").option("inferSchema", "true").csv('zones/')
df_zones.columns

In [ ]:
df_result = df_lagged.join(df_zones, df_lagged.zone == df_zones.LocationID)
df_result.select('hour', 'Zone', 'Borough', 'total_amount', 'prev_hour_amount', 'revenue_change') \
    .orderBy('zone', 'hour') \
    .show(20, truncate=False)

## 9. Write Final Result

In [ ]:
df_result.drop('LocationID', 'zone').write.parquet('tmp/revenue-zones-window', mode='overwrite')

In [ ]:
spark.version